<a href="https://colab.research.google.com/github/acastellanos-ie/NLP-MBDS-PT/blob/main/lingustic_structure_and_interpretability/stemming_lemmatization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 2: Deadly Information Retrieval

**Learning Objective:** Understand the mechanical and economic trade-offs of text standardization.

In class, we established that we need to "normalize" terms.

* **Stemming** chops off prefixes and suffixes to find a minimal unit of meaning. It handles matching "car" to "cars", but the result might not be a valid word.
* **Lemmatization** reduces inflections to their correct dictionary headword form. It also handles "car" to "cars" and "am" to "be", and must always return a valid word. However, lemmatization takes much more time than stemming.

Today, we will test whether that extra compute time is a luxury or a structural necessity.

In [4]:
# Phase 1: Environment Setup
# We need both NLTK (for classical stemming) and spaCy (for lemmatization)
!pip install -q spacy nltk
!python -m spacy download en_core_web_sm -q

import spacy
import nltk
nltk.download('punkt_tab')
from nltk.stem import PorterStemmer
import time

# Download the NLTK tokenizer dependency
nltk.download('punkt', quiet=True)

# Initialize models
nlp = spacy.load("en_core_web_sm")
stemmer = PorterStemmer()

print("Environment and models successfully initialized.")

✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


Environment and models successfully initialized.


### Phase 2: The Ambiguous Corpus

We have constructed a synthetic, highly ambiguous dataset. It contains sentences about two completely different domains that share similar morphological roots: Corporate Structuring and Human Biology.

In [2]:
# Our synthetic corpus
documents = [
    "The organization of the corporate body is strictly hierarchical.",
    "Organ donation is a critical procedure for the human body.",
    "Organizing the corporate body takes months of planning.",
    "The liver is a vital organ in the biological body."
]

# The query we want to resolve
query = "Organizing the corporate body"

print("Corpus loaded. Target Query:", query)

Corpus loaded. Target Query: Organizing the corporate body


### Phase 3: The Stemming Search Engine

We will build a basic search engine. First, we will use the `PorterStemmer`. Run the cell below to see what documents the stemmer retrieves when we search for *"Organizing the corporate body"*.

*Notice the execution time and the semantic accuracy of the results.*

In [5]:
class SearchEngine:
    def __init__(self, docs):
        self.docs = docs

    def process_stemming(self, text):
        # Tokenize and stem each word
        tokens = nltk.word_tokenize(text.lower())
        return [stemmer.stem(t) for t in tokens]

    def process_lemmatization(self, text):
        # 🛑 YOUR CHALLENGE: Implement this method using spaCy (self.nlp)
        # You must return a list of the lemmatized base forms of the text.
        pass

    def search_stemming(self, query):
        start = time.time()
        processed_query = set(self.process_stemming(query))

        # Print the stemmed query to audit the algorithmic destruction
        print(f"DEBUG - Stemmed Query: {processed_query}\n")

        results = []
        for doc in self.docs:
            processed_doc = set(self.process_stemming(doc))
            # Retrieve document if there is any vocabulary intersection
            if processed_query.intersection(processed_doc):
                results.append(doc)

        end = time.time()
        return results, (end - start) * 1000

    def search_lemmatization(self, query):
        start = time.time()
        processed_query = set(self.process_lemmatization(query))

        # Print the lemmatized query for auditing
        print(f"DEBUG - Lemmatized Query: {processed_query}\n")

        results = []
        for doc in self.docs:
            processed_doc = set(self.process_lemmatization(doc))
            if processed_query.intersection(processed_doc):
                results.append(doc)

        end = time.time()
        return results, (end - start) * 1000

# Execute the search using Stemming
engine = SearchEngine(documents)
engine.nlp = nlp # Pass the spacy model to the engine

res, exec_time = engine.search_stemming(query)
print(f"--- STEMMING RESULTS ---")
print(f"Execution Time: {exec_time:.4f} ms")
for r in res:
    print("-", r)

DEBUG - Stemmed Query: {'corpor', 'bodi', 'the', 'organ'}

--- STEMMING RESULTS ---
Execution Time: 27.0605 ms
- The organization of the corporate body is strictly hierarchical.
- Organ donation is a critical procedure for the human body.
- Organizing the corporate body takes months of planning.
- The liver is a vital organ in the biological body.


---
### 🛑 YOUR FORUM ASSIGNMENT

Did you notice that searching for "Organizing" retrieved documents about "Organ donation"? Look at the `DEBUG` output: the stemmer truncated "organizing" to "organ". It completely destroyed the semantic boundary between business and biology.

**Your Task:**
1. Complete the `process_lemmatization` method in the `SearchEngine` class above. Use `spaCy` to extract the proper dictionary lemma for each token.
2. Write the execution code to run `engine.search_lemmatization(query)` and store the retrieved results and the execution time.

**To submit in the forum:**
Post your `process_lemmatization` code and critically answer the following based on the empirical data you just generated:
* State the exact difference in milliseconds between executing the search with Stemming versus Lemmatization on your local machine.
* Assume you are the CTO of a legal tech firm processing 50 million documents a day. You have proven that lemmatization is semantically superior to stemming for this query. Based *strictly* on the latency difference you just recorded, does the computational cost justify the use of lemmatization in production, or would you settle for stemming to save server costs? Quantify your business logic.
*(Hint: An LLM cannot know the execution time on your hardware. Do the math).*